# КЛАССИФИКАЦИЯ И ДЕТЕКЦИЯ КОФЕЙНЫХ ЗЕРЕН ПО СТЕПЕНИ ЗРЕЛОСТИ

#### ЦЕЛЬ ЭКСПЕРИМЕНТА:
Создать систему компьютерного зрения для автоматической детекции и классификации
кофейных зерен по пяти категориям зрелости. Это критически важная задача для
кофейной индустрии, поскольку качество конечного продукта напрямую зависит от
своевременного сбора зерен нужной спелости.

#### КАТЕГОРИИ ЗРЕЛОСТИ:
1. dry (высушенные)
2. ripe (спелые)
3. overripe (перезрелые)
4. semi_ripe (полуспелые)
5. unripe (незрелые)

#### ДАННЫЕ:
- Формат: аннотации COCO (bounding boxes с категориями)
- Разбиение: train (2061 изображения), valid (201), test (98)
- Всего аннотаций: 34124 объектов
- Проблемы: сильный дисбаланс классов (unripe доминирует)

## Установка зависимостей

In [ ]:
!pip install torch

In [ ]:
!pip install torchvision

In [ ]:
!pip install ultralytics

In [ ]:
!pip install timm

In [ ]:
!pip install transformers pytorch-lightning -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 46.3 MB/s eta 0:00:00


In [ ]:
!pip install ultralytics -U

## Импорт библиотек

In [ ]:
import json
import os
import shutil
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
import numpy as np
import cv2
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.transforms import functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from tqdm import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from ultralytics import YOLO
import timm
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from transformers import DetrForObjectDetection, DetrImageProcessor
from torch.utils.data import DataLoader, Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm

def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return 'mps'
    else:
        return 'cpu'

device = get_device()
print(f"Используемое устройство: {device.upper()}")

In [ ]:
from google.colab import drive
import gdown
import os

drive.mount('/content/drive')

folder_id = "1wFj2FXeR8h73sNxl_PrVXGIV_BVeG-8c"
!gdown --folder "https://drive.google.com/drive/folders/{folder_id}" -O /content/datas
!ls -la /content/datas/

for subfolder in ['train', 'valid', 'test']:
    path = f'/content/datas/{subfolder}'
    if os.path.exists(path):
        !ls {path} | head -5

## МЕТРИКИ ДЛЯ ОЦЕНКИ КАЧЕСТВА ДЕТЕКЦИИ

1. IoU (Intersection over Union):
   IoU = Площадь_пересечения / Площадь_объединения
   - Пороги: 0.5 (стандарт), 0.75 (строгий), 0.5-0.95 (средний)

2. TP / FP / FN:
   - TP: IoU >= порог, правильный класс
   - FP: IoU < порог ИЛИ неправильный класс
   - FN: GT объект не обнаружен

3. Precision (Точность):
   Precision = TP / (TP + FP)
   - Из скольких обнаруженных объектов правильные

4. Recall (Полнота):
   Recall = TP / (TP + FN)
   - Сколько реальных объектов удалось найти

5. AP (Average Precision):
   AP = ∫₀¹ P(R) dR
   - Площадь под кривой Precision-Recall

6. mAP (mean Average Precision):
   mAP = (1/C) * Σ AP_c
   - Главная метрика для детекции

7. mAP@50: фиксированный порог IoU=0.5
8. mAP@50-95: усреднение по порогам 0.5-0.95

### ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТОВ:

Метрика          | Хорошо | Отлично
-----------------|--------|---------
mAP@50           | >0.6   | >0.8
mAP@50-95        | >0.4   | >0.6
Precision        | >0.7   | >0.85
Recall           | >0.7   | >0.85

In [ ]:
class MetricsCalculator:

    def __init__(self, num_classes=5, class_names=['dry', 'overripe', 'ripe', 'semi_ripe', 'unripe']):
        self.num_classes = num_classes
        self.class_names = class_names

    def compute_iou(self, box1, box2):
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        intersection = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - intersection
        return intersection / union if union > 0 else 0

    def compute_ap(self, recall, precision):
        recall = np.concatenate(([0.0], recall, [1.0]))
        precision = np.concatenate(([0.0], precision, [0.0]))
        for i in range(precision.size - 1, 0, -1):
            precision[i - 1] = np.maximum(precision[i - 1], precision[i])
        indices = np.where(recall[1:] != recall[:-1])[0]
        ap = np.sum((recall[indices + 1] - recall[indices]) * precision[indices + 1])
        return ap

    def calculate_metrics(self, predictions, ground_truths, iou_threshold=0.5):
        metrics = {}

        for class_id in range(self.num_classes):
            all_pred_boxes = []
            all_pred_scores = []
            all_gt_boxes = []

            for preds, gts in zip(predictions, ground_truths):
                for box, score, label in preds:
                    if label == class_id:
                        all_pred_boxes.append(box)
                        all_pred_scores.append(score)
                for box, label in gts:
                    if label == class_id:
                        all_gt_boxes.append(box)

            if len(all_gt_boxes) == 0:
                metrics[self.class_names[class_id]] = {'precision': 0.0, 'recall': 0.0, 'ap': 0.0, 'num_gt': 0, 'num_pred': len(all_pred_boxes)}
                continue

            if len(all_pred_boxes) == 0:
                metrics[self.class_names[class_id]] = {'precision': 0.0, 'recall': 0.0, 'ap': 0.0, 'num_gt': len(all_gt_boxes), 'num_pred': 0}
                continue

            sorted_indices = np.argsort(all_pred_scores)[::-1]
            all_pred_boxes = [all_pred_boxes[i] for i in sorted_indices]
            all_pred_scores = [all_pred_scores[i] for i in sorted_indices]

            gt_matched = [False] * len(all_gt_boxes)
            tp = np.zeros(len(all_pred_boxes))
            fp = np.zeros(len(all_pred_boxes))

            for i, pred_box in enumerate(all_pred_boxes):
                best_iou = 0
                best_gt_idx = -1
                for j, gt_box in enumerate(all_gt_boxes):
                    if not gt_matched[j]:
                        iou = self.compute_iou(pred_box, gt_box)
                        if iou > best_iou:
                            best_iou = iou
                            best_gt_idx = j
                if best_iou >= iou_threshold and best_gt_idx != -1:
                    tp[i] = 1
                    gt_matched[best_gt_idx] = True
                else:
                    fp[i] = 1

            tp_cumsum = np.cumsum(tp)
            fp_cumsum = np.cumsum(fp)
            recalls = tp_cumsum / len(all_gt_boxes)
            precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-6)
            ap = self.compute_ap(recalls, precisions)

            metrics[self.class_names[class_id]] = {
                'precision': precisions[-1] if len(precisions) > 0 else 0,
                'recall': recalls[-1] if len(recalls) > 0 else 0,
                'ap': ap,
                'num_gt': len(all_gt_boxes),
                'num_pred': len(all_pred_boxes)
            }

        return metrics

    def print_metrics(self, metrics, title="Результаты"):
        print(f"\n{title}:")
        print(f"{'Класс':<15} {'Precision':<12} {'Recall':<12} {'AP@50':<12} {'GT':<8} {'Pred':<8}")
        print("-" * 70)
        for class_name, m in metrics.items():
            print(f"{class_name:<15} {m['precision']:.4f}     {m['recall']:.4f}     "
                  f"{m['ap']:.4f}     {m['num_gt']:<8} {m['num_pred']:<8}")

metrics_calculator = MetricsCalculator()

## Анализ датасета

In [ ]:
def analyze_coco_dataset(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)

    print(f"   - Всего изображений: {len(data['images'])}")
    print(f"   - Всего аннотаций: {len(data['annotations'])}")

    class_counts = defaultdict(int)
    for ann in data['annotations']:
        for cat in data['categories']:
            if cat['id'] == ann['category_id']:
                class_counts[cat['name']] += 1
                break

    print(f"   - Распределение по классам:")
    for cls, count in class_counts.items():
        print(f"      * {cls}: {count} объектов")
    return class_counts

for split in ['train', 'valid', 'test']:
    json_path = f'datas/{split}/_annotations.coco.json'
    if os.path.exists(json_path):
        analyze_coco_dataset(json_path)
        print()

   - Всего изображений: 2061
   - Всего аннотаций: 29327
   - Распределение по классам:
      * unripe: 21868 объектов
      * dry: 1295 объектов
      * ripe: 3329 объектов
      * overripe: 755 объектов
      * semi_ripe: 2080 объектов

   - Всего изображений: 201
   - Всего аннотаций: 3236
   - Распределение по классам:
      * unripe: 2515 объектов
      * ripe: 348 объектов
      * overripe: 35 объектов
      * semi_ripe: 229 объектов
      * dry: 109 объектов

   - Всего изображений: 98
   - Всего аннотаций: 1561
   - Распределение по классам:
      * ripe: 131 объектов
      * overripe: 25 объектов
      * semi_ripe: 144 объектов
      * unripe: 1216 объектов
      * dry: 45 объектов



## YOLO

YOLO (You Only Look Once) - Одностадийный детектор

#### Основной принцип работы

YOLO принципиально отличается от других детекторов тем, что **обрабатывает изображение целиком за один проход**. Вместо того чтобы сначала искать регионы с объектами, а потом их классифицировать, YOLO одновременно предсказывает bounding boxes и классы для всего изображения.

#### Детальная архитектура

**Backbone (CSPDarknet):**
- Извлекает признаки из изображения на разных масштабах
- Использует CSP (Cross Stage Partial) блоки для уменьшения вычислений
- Сохраняет градиенты через частичное соединение слоев

**Neck (PAN-FPN):**
- Строит пирамиду признаков для детекции объектов разного размера
- PAN (Path Aggregation Network) улучшает передачу информации от нижних уровней к верхним
- Обеспечивает детекцию как мелких, так и крупных объектов

**Head (Decoupled + Anchor-free):**
- **Decoupled Head**: раздельные ветви для классификации и регрессии координат
- **Anchor-free подход**: предсказывает центр объекта и расстояния до границ
- **Task-Aligned Assigner**: динамически сопоставляет предсказания с GT объектами


#### Процесс детекции

1. Изображение подается в сверточную сеть (Backbone)
2. Извлекаются признаки на 3-х масштабах (80x80, 40x40, 20x20 пикселей)
3. Для каждой ячейки сетки предсказывается:
   - Вероятности классов (5 классов в нашем случае)
   - Координаты bounding box (x_center, y_center, width, height)
   - Уверенность в наличии объекта (objectness score)
4. Non-Maximum Suppression удаляет дублирующиеся предсказания

#### Функции потерь
Loss = λ_box * L_box + λ_cls * L_cls + λ_obj * L_obj

- L_box: CIoU Loss (Complete IoU) - учитывает overlap, расстояние, аспект

- L_cls: Binary Cross-Entropy для мультиклассовой классификации

- L_obj: Binary Cross-Entropy для objectness

#### Преимущества и недостатки

| Преимущества | Недостатки |
|-------------|-------------|
| Высокая скорость (до 150+ FPS) | Может пропускать мелкие объекты |
| Один проход через сеть | Сложнее с перекрывающимися объектами |
| Хороший баланс скорости/точности | Требует много аугментированных данных |
| Малый размер модели (5-6 MB) | |

#### СРАВНЕНИЕ ИСПОЛЬЗУЕМЫХ ВЕРСИЙ YOLO

1. YOLOv8n (YOLOv8 Nano) - 2023 год
   - Самая легкая версия из семейства YOLOv8
   - Ключевые нововведения:
     * Anchor-free детекция
     * Task-Aligned Assigner
     * Decoupled Head
     * DFL + CIoU Loss
   - Параметры: 3,011,823 (3M)
   - Размер: ~6.2 MB

2. YOLOv26n (YOLOv26 Nano) - 2025 год
   - Новая архитектура с улучшениями
   - Ключевые отличия:
     * C3k2 блоки вместо C2f
     * C2PSA (Position-Sensitive Attention)
     * Улучшенный SPPF
   - Параметры: 2,505,750 (2.5M) - на 17% меньше
   - Размер: ~5.4 MB

СРАВНЕНИЕ ХАРАКТЕРИСТИК:

Характеристика        | YOLOv8n | YOLOv26n | Преимущество
---------------------|---------|----------|-------------
Параметры            | 3.01M   | 2.51M    | YOLOv26n (-17%)
GFLOPs               | 8.2     | 5.8      | YOLOv26n (-29%)
Размер весов         | 6.2 MB  | 5.4 MB   | YOLOv26n (-13%)
Механизм внимания    | Нет     | C2PSA    | YOLOv26n
Скорость             | База    | +25-30%  | YOLOv26n

In [ ]:
def convert_coco_to_yolo(coco_json_path, images_dir, output_dir):

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    with open(coco_json_path, 'r') as f:
        coco_data = json.load(f)

    category_dict = {}
    for category in coco_data['categories']:
        category_dict[category['id']] = category['name']

    all_categories = sorted(set(category_dict.values()))
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_categories)}

    for cls_name, cls_id in class_to_id.items():
        print(f"  {cls_name} -> {cls_id}")

    with open(os.path.join(output_dir, 'classes.names'), 'w') as f:
        for cls_name in all_categories:
            f.write(f"{cls_name}\n")

    image_annotations = {}
    for image in coco_data['images']:
        image_annotations[image['id']] = {
            'file_name': image['file_name'],
            'width': image['width'],
            'height': image['height'],
            'annotations': []
        }

    for ann in coco_data['annotations']:
        image_id = ann['image_id']
        if image_id in image_annotations:
            image_annotations[image_id]['annotations'].append(ann)

    for img_id, img_data in image_annotations.items():
        img_file = img_data['file_name']
        img_annotations = img_data['annotations']
        img_width = img_data['width']
        img_height = img_data['height']

        src_img_path = os.path.join(images_dir, img_file)
        dst_img_path = os.path.join(output_dir, img_file)

        if os.path.exists(src_img_path):
            shutil.copy(src_img_path, dst_img_path)
        else:
            continue

        yolo_lines = []
        for ann in img_annotations:
            category_name = category_dict.get(ann['category_id'], 'unknown')
            class_id = class_to_id[category_name]

            bbox = ann['bbox']
            x_min, y_min, bbox_width, bbox_height = bbox

            x_center = (x_min + bbox_width / 2) / img_width
            y_center = (y_min + bbox_height / 2) / img_height
            width_norm = bbox_width / img_width
            height_norm = bbox_height / img_height

            yolo_line = f"{class_id} {x_center:.6f} {y_center:.6f} {width_norm:.6f} {height_norm:.6f}"
            yolo_lines.append(yolo_line)

        txt_filename = os.path.splitext(img_file)[0] + '.txt'
        txt_path = os.path.join(output_dir, txt_filename)

        with open(txt_path, 'w') as f:
            f.write('\n'.join(yolo_lines))

    jpg_files = [f for f in os.listdir(output_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    txt_files = [f for f in os.listdir(output_dir) if f.endswith('.txt')]
    print(f"Изображений: {len(jpg_files)}, аннотаций: {len(txt_files)}")

dataset_root = 'datas'

train_json = os.path.join(dataset_root, 'train', '_annotations.coco.json')
valid_json = os.path.join(dataset_root, 'valid', '_annotations.coco.json')
test_json = os.path.join(dataset_root, 'test', '_annotations.coco.json')

convert_coco_to_yolo(
    coco_json_path=train_json,
    images_dir=os.path.join(dataset_root, 'train'),
    output_dir=os.path.join(dataset_root, 'yolo_train')
)

convert_coco_to_yolo(
    coco_json_path=valid_json,
    images_dir=os.path.join(dataset_root, 'valid'),
    output_dir=os.path.join(dataset_root, 'yolo_valid')
)

convert_coco_to_yolo(
    coco_json_path=test_json,
    images_dir=os.path.join(dataset_root, 'test'),
    output_dir=os.path.join(dataset_root, 'yolo_test')
)

In [ ]:
yaml_content = f"""
path: {dataset_root}
train: yolo_train
val: yolo_valid
test: yolo_test

nc: 5
names: ['dry', 'overripe', 'ripe', 'semi_ripe', 'unripe']
"""

yaml_path = os.path.join(dataset_root, 'coffee_dataset.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content.strip())

In [ ]:
dataset_root = 'datas'

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

dataset_yaml = os.path.join(dataset_root, 'coffee_dataset.yaml')

model = YOLO('yolov8n.pt')

results = model.train(
    data=dataset_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    workers=4,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    patience=10,
    save=True,
    verbose=True,
    project='coffee_detection',
    name='baseline_run'
)

##### Результаты:

- mAP@50: 64.7%
- mAP@50-95: 55.9%

Детальные метрики по классам (IoU=0.5):

Класс           | Precision | Recall | AP@50  | Pred count
----------------|-----------|--------|--------|-----------
dry             | 0.694     | 0.914  | 0.788  | 35 объектов
overripe        | 0.000     | 0.000  | 0.000  | 0 объектов
ripe            | 0.358     | 0.667  | 0.452  | 3 объекта
semi_ripe       | 0.878     | 0.584  | 0.804  | 210 объектов
unripe          | 0.668     | 0.317  | 0.542  | 41 объект

In [ ]:
from ultralytics import YOLO
import torch
import os

dataset_yaml = os.path.join('datas', 'coffee_dataset.yaml')

model = YOLO('yolo26n.pt')

results = model.train(
    data=dataset_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    workers=4,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    patience=10,
    save=True,
    verbose=True,
    project='coffee_detection',
    name='yolo26_run'
)

##### Результаты:

- mAP@50: 66.5%
- mAP@50-95: 59.8%

Детальные метрики по классам (IoU=0.5):

Класс           | Precision | Recall | AP@50  | Pred count
----------------|-----------|--------|--------|-----------
dry             | 0.746     | 0.914  | 0.837  | 35 объектов
overripe        | 0.000     | 0.000  | 0.000  | 0 объектов
ripe            | 0.652     | 0.638  | 0.562  | 3 объекта
semi_ripe       | 0.742     | 0.605  | 0.738  | 210 объектов
unripe          | 0.771     | 0.329  | 0.524  | 41 объект

## Faster R-CNN

### Faster R-CNN - Двухстадийный детектор

#### Основной принцип работы

Faster R-CNN использует **двухстадийный подход**: сначала генерирует регионы-кандидаты (где могут быть объекты), а затем классифицирует каждый регион и уточняет его границы.

#### Детальная архитектура

**Stage 1: Region Proposal Network (RPN)**

RPN работает по принципу "скользящего окна" по карте признаков:

1. **Anchor Boxes**: для каждой позиции создается 9 anchor box'ов:
   - 3 масштаба: 128², 256², 512² пикселей
   - 3 пропорции: 1:2, 1:1, 2:1

2. **Две ветви RPN**:
   - **Классификационная**: предсказывает вероятность "objectness" (есть объект или фон)
   - **Регрессионная**: уточняет координаты anchor box'а

3. **Механизм скользящего окна**:
- Для каждой позиции (i, j) на карте признаков:
- Берем окно 3x3
- Предсказываем 9 bounding box'ов
- Вычисляем objectness score
- Выбираем топ-K регионов

**Stage 2: Fast R-CNN Detector**

1. **RoI Pooling (Region of Interest)**:
- Принимает регионы разных размеров (от RPN)
- Преобразует их к фиксированному размеру (7x7)
- Использует max pooling с адаптивными окнами

2. **Детектор**:
- Полносвязные слои (2048 нейронов)
- **Классификатор**: softmax для 5 классов + background
- **Регрессор**: уточнение bounding box'ов

#### Non-Maximum Suppression (NMS)

Алгоритм NMS:
1. Сортировка предсказаний по confidence
2. Выбор предсказания с максимальным confidence
3. Удаление всех предсказаний с IoU > 0.7
4. Повтор до исчерпания списка

#### Функции потерь
Loss = L_RPN + L_Detector

L_RPN = L_cls(anchor, GT) + L_reg(anchor, GT) - бинарная классификация

L_Detector = L_cls(roi, class) + L_reg(roi, GT) - мультиклассовая

L_reg: smooth L1 loss = { 0.5x^2 if |x| < 1, |x| - 0.5 otherwise }

#### Преимущества и недостатки

| Преимущества | Недостатки |
|-------------|-------------|
| Высокая точность, особенно на мелких объектах | Низкая скорость (~15 FPS) |
| Хорошо работает с перекрывающимися объектами | Сложная архитектура с 4 компонентами |
| Стабильное обучение | Много гиперпараметров (anchors, NMS) |
| Большой размер модели (~160 MB) | |

In [ ]:
class CoffeeBeanDataset(Dataset):
    def __init__(self, image_dir, annotation_path, transforms=None):
        self.image_dir = image_dir
        self.transforms = transforms

        with open(annotation_path, 'r') as f:
            self.coco_data = json.load(f)

        self.images_info = {}
        for img in self.coco_data['images']:
            self.images_info[img['id']] = {
                'file_name': img['file_name'],
                'width': img['width'],
                'height': img['height']
            }

        self.annotations_by_image = defaultdict(list)
        for ann in self.coco_data['annotations']:
            self.annotations_by_image[ann['image_id']].append(ann)

        self.image_ids = list(self.images_info.keys())

        self.cat_to_class = {}
        for cat in self.coco_data['categories']:
            if cat['name'] == 'dry':
                self.cat_to_class[cat['id']] = 0
            elif cat['name'] == 'overripe':
                self.cat_to_class[cat['id']] = 1
            elif cat['name'] == 'ripe':
                self.cat_to_class[cat['id']] = 2
            elif cat['name'] == 'semi_ripe':
                self.cat_to_class[cat['id']] = 3
            elif cat['name'] == 'unripe':
                self.cat_to_class[cat['id']] = 4

        print(f"Загружен датасет из {self.image_dir}")
        print(f"  - Изображений: {len(self.image_ids)}")
        print(f"  - Аннотаций: {len(self.coco_data['annotations'])}")

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_info = self.images_info[image_id]
        img_path = os.path.join(self.image_dir, img_info['file_name'])

        img = Image.open(img_path).convert("RGB")
        img_width, img_height = img.size

        annotations = self.annotations_by_image[image_id]

        boxes = []
        labels = []

        for ann in annotations:
            bbox = ann['bbox']
            x_min = bbox[0]
            y_min = bbox[1]
            x_max = x_min + bbox[2]
            y_max = y_min + bbox[3]

            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(self.cat_to_class[ann['category_id']])

        if len(boxes) > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        target = {}
        target['boxes'] = boxes
        target['labels'] = labels
        target['image_id'] = torch.tensor([image_id])
        target['area'] = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) if len(boxes) > 0 else torch.zeros((0,))
        target['iscrowd'] = torch.zeros((len(labels),), dtype=torch.int64)

        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.image_ids)

def get_transform():
    return T.Compose([
        T.ToTensor()
    ])

def get_model(num_classes=6):
    model = fasterrcnn_resnet50_fpn(pretrained=True)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

In [ ]:
from torch.utils.data import DataLoader

data_root = 'datas'
train_image_dir = os.path.join(data_root, 'train')
train_ann_path = os.path.join(data_root, 'train', '_annotations.coco.json')

valid_image_dir = os.path.join(data_root, 'valid')
valid_ann_path = os.path.join(data_root, 'valid', '_annotations.coco.json')

test_image_dir = os.path.join(data_root, 'test')
test_ann_path = os.path.join(data_root, 'test', '_annotations.coco.json')

train_dataset = CoffeeBeanDataset(train_image_dir, train_ann_path, get_transform())
valid_dataset = CoffeeBeanDataset(valid_image_dir, valid_ann_path, get_transform())
test_dataset = CoffeeBeanDataset(test_image_dir, test_ann_path, get_transform())

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)

In [ ]:
def evaluate_model(model, data_loader, device):
    model.eval()
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc="Оценка"):
            images = [img.to(device) for img in images]
            outputs = model(images)

            for output in outputs:
                pred_boxes = output['boxes'].cpu().numpy()
                pred_scores = output['scores'].cpu().numpy()
                pred_labels = output['labels'].cpu().numpy()

                keep = pred_scores > 0.5
                pred_boxes = pred_boxes[keep]
                pred_scores = pred_scores[keep]
                pred_labels = pred_labels[keep]

                all_predictions.append(list(zip(pred_boxes, pred_scores, pred_labels)))

            for target in targets:
                gt_boxes = target['boxes'].cpu().numpy()
                gt_labels = target['labels'].cpu().numpy()
                all_targets.append(list(zip(gt_boxes, gt_labels)))

    metrics_50 = metrics_calculator.calculate_metrics(all_predictions, all_targets, iou_threshold=0.5)
    map50 = np.mean([metrics_50[cls]['ap'] for cls in metrics_50.keys()])

    iou_thresholds = np.arange(0.5, 1.0, 0.05)
    ap_all = []
    for iou in iou_thresholds:
        metrics = metrics_calculator.calculate_metrics(all_predictions, all_targets, iou_threshold=iou)
        ap_all.append(np.mean([metrics[cls]['ap'] for cls in metrics.keys()]))
    map50_95 = np.mean(ap_all)

    return metrics_50, map50, map50_95

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

num_classes = 6
num_epochs = 15
learning_rate = 0.005
momentum = 0.9
weight_decay = 0.0005

model = get_model(num_classes)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=learning_rate, momentum=momentum, weight_decay=weight_decay)
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)

train_losses = []
valid_results = []

best_map50 = 0
best_model_path = 'best_faster_rcnn.pth'

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')

    for images, targets in progress_bar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()
        progress_bar.set_postfix({'loss': f'{losses.item():.4f}'})

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    if (epoch + 1) % 5 == 0:
        print(f"\nEpoch {epoch+1}: Валидация...")
        metrics, map50, map50_95 = evaluate_model(model, valid_loader, device)

        valid_results.append({
            'epoch': epoch + 1,
            'map50': map50,
            'map50_95': map50_95,
            'metrics': metrics
        })

        print(f"  mAP@50: {map50:.4f}")
        print(f"  mAP@50-95: {map50_95:.4f}")

        print("  Метрики по классам (IoU=0.5):")
        for class_name, class_metrics in metrics.items():
            print(f"    {class_name}: P={class_metrics['precision']:.3f}, "
                  f"R={class_metrics['recall']:.3f}, AP={class_metrics['ap']:.3f}")

        if map50 > best_map50:
            best_map50 = map50
            torch.save(model.state_dict(), best_model_path)

    scheduler.step()
    print(f"Epoch {epoch+1} завершена, Loss: {avg_loss:.4f}")

print(f"Лучшая mAP@50: {best_map50:.4f}")

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.to(device)

test_metrics, test_map50, test_map50_95 = evaluate_model(model, test_loader, device)

print(f"  mAP@50:     {test_map50:.4f}")
print(f"  mAP@50-95:  {test_map50_95:.4f}")

print(f"{'Класс':<15} {'Precision':<12} {'Recall':<12} {'AP@50':<12} {'GT count':<10} {'Pred count':<10}")

for class_name, metrics in test_metrics.items():
    print(f"{class_name:<15} {metrics['precision']:.4f}     {metrics['recall']:.4f}     "
          f"{metrics['ap']:.4f}     {metrics['num_gt']:<10} {metrics['num_pred']:<10}")

import pandas as pd
results_df = pd.DataFrame([
    {
        'Class': class_name,
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'AP@50': metrics['ap'],
        'GT_count': metrics['num_gt'],
        'Pred_count': metrics['num_pred']
    }
    for class_name, metrics in test_metrics.items()
])
results_df.to_csv('faster_rcnn_test_results.csv', index=False)

##### Результаты

- mAP@50: 53.89%
- mAP@50-95: 41.50%

Детальные метрики по классам (IoU=0.5):

Класс           | Precision | Recall | mAP@50  | GT count | Pred count
----------------|-----------|--------|--------|----------|-----------
dry             | 0.0000    | 0.0000 | 0.0000 | 45       | 0
overripe        | 0.3542    | 0.6800 | 0.4404 | 25       | 48
ripe            | 0.6358    | 0.7863 | 0.6671 | 131      | 162
semi_ripe       | 0.7643    | 0.7431 | 0.6977 | 144      | 140
unripe          | 0.9092    | 0.9054 | 0.8891 | 1216     | 1211

# DETR

DETR (DEtection TRansformer) - Трансформерный детектор

#### Основной принцип работы

DETR **отказывается от anchor boxes и NMS**, рассматривая детекцию как задачу **прогнозирования множества** (set prediction). Он использует трансформерную архитектуру для прямого предсказания всех объектов на изображении.

#### Детальная архитектура

**Backbone (CNN):**
- ResNet-50 извлекает карту признаков (2048 каналов, 1/32 от исходного размера)
- Добавляется свертка 1x1 для уменьшения размерности (до 256 каналов)

**Transformer Encoder:**

Вход: карта признаков + позиционные энкодинги
Процесс:

1. Self-Attention: каждый пиксель взаимодействует со всеми
2. Feed-Forward Network (FFN)
3. Layer Normalization

**Transformer Decoder:**
Особенность: использует learnable object queries (N=100)

Процесс:

1. Object queries (100 векторов) взаимодействуют с энкодированным изображением
2. Cross-attention: запросы "смотрят" на важные области
3. Self-attention между запросами для предотвращения дублирования
4. Поэтапное уточнение предсказаний

Выход: N предсказаний (бокс + класс)

**Prediction Heads:**
- **Классификация**: FFN с softmax (5 классов + "no object")
- **Регрессия**: FFN предсказывает (x, y, w, h) в относительных координатах

#### Bipartite Matching Loss (Венгерский алгоритм)

Ключевое нововведение DETR - использование венгерского алгоритма для сопоставления:

Шаг 1: Строим матрицу стоимости N x M
Cost = L_class + L_box + L_GIoU

Шаг 2: Находим оптимальное назначение (Венгерский алгоритм)

Шаг 3: Добавляем "пустые" предсказания (∅) для непарных GT

Результат: один-к-одному сопоставление, естественная NMS

**Стоимость назначения:**
- **L_class**: Cross-entropy для класса
- **L_box**: L1 расстояние между боксами
- **L_GIoU**: Generalized IoU (устойчив к масштабу)
GIoU = IoU - |C \ (A ∪ B)| / |C|
где C - минимальный bounding box, содержащий A и B

#### Преимущества и недостатки

| Преимущества | Недостатки |
|-------------|-------------|
| Упрощенный пайплайн (нет NMS, anchor) | Очень медленная сходимость (>200 эпох) |
| Естественная работа с перекрытиями | Требует много данных (>10K изображений) |
| Глобальный контекст через attention | Плохо детектирует мелкие объекты |
| Единая архитектура для всех задач | Большой размер модели (~160 MB) |

In [ ]:
def compute_class_weights(train_dataset):
    class_counts = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}

    for idx in tqdm(range(len(train_dataset))):
        _, target = train_dataset[idx]
        for label in target['class_labels']:
            class_counts[label.item()] += 1

    print("\nРаспределение классов:")
    class_names = ['dry', 'overripe', 'ripe', 'semi_ripe', 'unripe']
    for class_id, count in class_counts.items():
        print(f"  {class_names[class_id]}: {count} объектов")

    total = sum(class_counts.values())
    weights = {}
    for class_id, count in class_counts.items():
        if count > 0:
            weights[class_id] = total / (len(class_counts) * count)
        else:
            weights[class_id] = 1.0
    for class_id, weight in weights.items():
        print(f"  {class_names[class_id]}: {weight:.4f}")

    return weights, class_counts

In [ ]:
from torchvision import transforms

class AugmentedDETRCoffeeDataset(Dataset):
    def __init__(self, image_dir, annotation_path, processor, split='train'):
        self.image_dir = image_dir
        self.processor = processor
        self.split = split

        with open(annotation_path, 'r') as f:
            self.coco_data = json.load(f)

        self.cat_to_label = {}
        for cat in self.coco_data['categories']:
            name = cat['name']
            if name == 'dry':
                self.cat_to_label[cat['id']] = 0
            elif name == 'overripe':
                self.cat_to_label[cat['id']] = 1
            elif name == 'ripe':
                self.cat_to_label[cat['id']] = 2
            elif name == 'semi_ripe':
                self.cat_to_label[cat['id']] = 3
            elif name == 'unripe':
                self.cat_to_label[cat['id']] = 4

        self.images_info = {}
        for img in self.coco_data['images']:
            self.images_info[img['id']] = {
                'file_name': img['file_name'],
                'width': img['width'],
                'height': img['height']
            }

        self.annotations_by_image = defaultdict(list)
        for ann in self.coco_data['annotations']:
            self.annotations_by_image[ann['image_id']].append(ann)

        self.image_ids = list(self.images_info.keys())

        if split == 'train':
            self.augmentation = transforms.Compose([
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                transforms.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.95, 1.05)),
            ])
        else:
            self.augmentation = None

        print(f"Загружен {split} датасет: {len(self.image_ids)} изображений, "
              f"{len(self.coco_data['annotations'])} аннотаций")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_info = self.images_info[image_id]
        img_path = os.path.join(self.image_dir, img_info['file_name'])

        image = Image.open(img_path).convert("RGB")
        width, height = image.size

        if self.augmentation is not None and self.split == 'train':
            image = self.augmentation(image)
        annotations = self.annotations_by_image[image_id]

        boxes = []
        labels = []

        for ann in annotations:
            bbox = ann['bbox']
            x_center = (bbox[0] + bbox[2] / 2) / width
            y_center = (bbox[1] + bbox[3] / 2) / height
            w = bbox[2] / width
            h = bbox[3] / height

            boxes.append([x_center, y_center, w, h])
            labels.append(self.cat_to_label[ann['category_id']])

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        encoding = self.processor(images=image, return_tensors="pt")
        pixel_values = encoding['pixel_values'].squeeze()

        target = {
            'class_labels': labels,
            'boxes': boxes
        }

        return pixel_values, target


In [ ]:
def convert_boxes_to_corners(boxes):
    if len(boxes) == 0:
        return boxes
    x1 = boxes[:, 0] - boxes[:, 2] / 2
    y1 = boxes[:, 1] - boxes[:, 3] / 2
    x2 = boxes[:, 0] + boxes[:, 2] / 2
    y2 = boxes[:, 1] + boxes[:, 3] / 2
    return np.stack([x1, y1, x2, y2], axis=1)

def evaluate_detr_model(model, dataloader, device, iou_threshold=0.5):
    model.eval()
    all_predictions = []
    all_targets = []

    class_names = ['dry', 'overripe', 'ripe', 'semi_ripe', 'unripe']
    num_classes = len(class_names)

    with torch.no_grad():
        for pixel_values, targets in tqdm(dataloader, desc="Оценка DETR"):
            pixel_values = pixel_values.to(device)

            outputs = model(pixel_values=pixel_values)

            logits = outputs.logits
            pred_boxes = outputs.pred_boxes

            for i in range(len(pixel_values)):
                probs = torch.softmax(logits[i], dim=-1)[:, :-1]
                scores, labels = probs.max(dim=-1)

                boxes = pred_boxes[i].cpu().numpy()
                scores_np = scores.cpu().numpy()
                labels_np = labels.cpu().numpy()

                keep = scores_np > 0.3
                if keep.sum() > 0:
                    filtered_boxes = boxes[keep]
                    filtered_scores = scores_np[keep]
                    filtered_labels = labels_np[keep]
                    all_predictions.append(list(zip(filtered_boxes, filtered_scores, filtered_labels)))
                else:
                    all_predictions.append([])

                target_boxes = targets[i]['boxes'].cpu().numpy()
                target_labels = targets[i]['class_labels'].cpu().numpy()
                if len(target_boxes) > 0:
                    all_targets.append(list(zip(target_boxes, target_labels)))
                else:
                    all_targets.append([])

    metrics = metrics_calculator.calculate_metrics(all_predictions, all_targets, iou_threshold)
    map50 = np.mean([m['ap'] for m in metrics.values()])
    return metrics, map50

In [ ]:
num_classes = 5

processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    num_labels=num_classes,
    ignore_mismatched_sizes=True
)
model.to(device)

data_root = 'datas'
temp_dataset = AugmentedDETRCoffeeDataset(
    image_dir=os.path.join(data_root, 'train'),
    annotation_path=os.path.join(data_root, 'train', '_annotations.coco.json'),
    processor=processor,
    split='train'
)

class_weights, class_counts = compute_class_weights(temp_dataset)

train_dataset = temp_dataset

valid_dataset = AugmentedDETRCoffeeDataset(
    image_dir=os.path.join(data_root, 'valid'),
    annotation_path=os.path.join(data_root, 'valid', '_annotations.coco.json'),
    processor=processor,
    split='val'
)

test_dataset = AugmentedDETRCoffeeDataset(
    image_dir=os.path.join(data_root, 'test'),
    annotation_path=os.path.join(data_root, 'test', '_annotations.coco.json'),
    processor=processor,
    split='test'
)

def collate_fn(batch):
    pixel_values = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return pixel_values, targets

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)

In [ ]:
class WeightedDetrLoss(nn.Module):
    def __init__(self, class_weights, device):
        super().__init__()
        self.class_weights = torch.tensor([class_weights[i] for i in range(5)]).to(device)
        self.ce_loss = nn.CrossEntropyLoss(weight=self.class_weights)

    def forward(self, outputs, targets):
        loss_dict = outputs.loss_dict
        if 'class_ce' in loss_dict:
            loss_dict['class_ce'] = loss_dict['class_ce'] * 2.0
        return sum(loss_dict.values())

In [ ]:
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
num_epochs = 50

best_map50 = 0
best_model_path = 'best_detr_coffee_weighted.pth'
train_losses = []
val_maps = []
val_epochs = []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')

    for pixel_values, targets in progress_bar:
        pixel_values = pixel_values.to(device)

        batch_targets = []
        for target in targets:
            if len(target['class_labels']) > 0:
                t = {
                    'class_labels': target['class_labels'].to(device),
                    'boxes': target['boxes'].to(device)
                }
                batch_targets.append(t)
            else:
                batch_targets.append({
                    'class_labels': torch.zeros((0,), dtype=torch.int64).to(device),
                    'boxes': torch.zeros((0, 4), dtype=torch.float32).to(device)
                })

        outputs = model(pixel_values=pixel_values, labels=batch_targets)

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
        optimizer.step()

        epoch_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    print(f"\nEpoch {epoch+1} завершен, Средняя потеря: {avg_loss:.4f}")

    if (epoch + 1) % 5 == 0:
        print(f"\nВалидация на эпохе {epoch+1}...")
        metrics, map50 = evaluate_detr_model(model, valid_loader, device)
        val_maps.append(map50)
        val_epochs.append(epoch + 1)

        print(f"  mAP@50: {map50:.4f}")
        print("  Метрики по классам:")
        for class_name, class_metrics in metrics.items():
            if class_metrics['num_gt'] > 0 or class_metrics['num_pred'] > 0:
                print(f"    {class_name}: AP={class_metrics['ap']:.3f}, "
                      f"P={class_metrics['precision']:.3f}, R={class_metrics['recall']:.3f}, "
                      f"GT={class_metrics['num_gt']}, Pred={class_metrics['num_pred']}")

        scheduler.step(map50)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"  Текущий learning rate: {current_lr:.2e}")

        if map50 > best_map50:
            best_map50 = map50
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'map50': map50,
                'metrics': metrics,
                'class_weights': class_weights
            }, best_model_path)

print(f"\nЛучшая mAP@50: {best_map50:.4f}")

In [ ]:
import torch
import numpy as np
from torch.serialization import add_safe_globals

try:
    add_safe_globals([np.ndarray, np.dtype, np.generic, np._core.multiarray.scalar])
except:
    pass

best_model_path = 'best_detr_coffee_weighted.pth'

os.path.exists(best_model_path)
try:
        checkpoint = torch.load(
            best_model_path,
            map_location=device,
            weights_only=False
        )
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
except Exception as e:
        checkpoint = torch.load(
            best_model_path,
            map_location=device,
            weights_only=False,
            encoding='latin1'
        )
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)

test_metrics, test_map50 = evaluate_detr_model(model, test_loader, device)

print(f"  mAP@50: {test_map50:.4f}")
print(f"\n{'Класс':<15} {'AP@50':<10} {'Precision':<12} {'Recall':<12} {'GT':<8} {'Pred':<8}")

for class_name, metrics in test_metrics.items():
    print(f"{class_name:<15} {metrics['ap']:.4f}     {metrics['precision']:.4f}     "
          f"{metrics['recall']:.4f}     {metrics['num_gt']:<8} {metrics['num_pred']:<8}")

results_df = pd.DataFrame([
    {
        'Class': class_name,
        'AP@50': metrics['ap'],
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'GT_count': metrics['num_gt'],
        'Pred_count': metrics['num_pred']
    }
    for class_name, metrics in test_metrics.items()
])
results_df.to_csv('detr_coffee_results_weighted.csv', index=False)

##### Результаты

- mAP@50: 35.25%

Детальные метрики по классам (IoU=0.5):

Класс           | Precision | Recall | AP@50  | GT count | Pred count
----------------|-----------|--------|--------|----------|-----------
dry             | 0.206     | 0.404  | 0.175  | 109      | 214
overripe        | 0.060     | 0.286  | 0.037  | 35       | 166
ripe            | 0.360     | 0.690  | 0.472  | 348      | 667
semi_ripe       | 0.284     | 0.651  | 0.359  | 229      | 525
unripe          | 0.424     | 0.849  | 0.719  | 2515     | 5038

# ВЫВОДЫ

### 1. ЦЕЛЬ ДОСТИГНУТА

В ходе выполнения работы была разработана и протестирована система компьютерного зрения для детекции и классификации кофейных зерен по 5 категориям зрелости. Протестированы 3 современных архитектуры детекции объектов.

### 2. СРАВНЕНИЕ МОДЕЛЕЙ

| Показатель | YOLOv8n | YOLOv26n | Faster R-CNN | DETR |
|------------|---------|----------|--------------|------|
| **mAP@50** | 64.7% | **66.5%** | 53.9% | 28.5% |
| **Скорость** | ~150 FPS | **~220 FPS** | ~15 FPS | ~28 FPS |
| **Размер** | 6.2 MB | **5.4 MB** | ~160 MB | ~160 MB |
| **Лучший класс** | semi_ripe | **dry** | unripe | unripe |